# Motion Detection Using Recursive Moving Average and Richefeu’s Algorithm

Motion detection plays a crucial role in various computer vision applications, including surveillance, object tracking, and anomaly detection. In this notebook, we explore two distinct motion detection methods:  

1. **Recursive Moving Average (RMA):** A background subtraction technique that iteratively updates a static background model using an exponential moving average.  
2. **Richefeu’s Algorithm:** A morphological-based motion detection approach that employs adaptive dilation and erosion to capture dynamic foreground changes.  

Each method is implemented and analyzed to highlight its effectiveness in different motion detection scenarios.  

In [26]:
import cv2
import numpy as np
import imutils
from IPython.display import HTML

## 1. Video Loading and Display

In [ ]:
video_path = "data/168811-839864556_tiny.mp4"

# Read video file
cap = cv2.VideoCapture(video_path)

# Check if video file is opened successfully
if not cap.isOpened():
    print("Error opening video file")  

# create the HTML code to display the video
html_code = f"""
<video width="640" height="480" controls>
  <source src="{video_path}" type="video/mp4">
</video>
"""

# display the HTML code in the notebook cell
HTML(html_code)

## 2. Recursive Moving Average for Motion Detection
The Recursive Moving Average (RMA) method is a simple yet effective technique for detecting motion in a video sequence. It works by maintaining an exponentially weighted average of previous frames and comparing it to the current frame to detect changes. This approach is particularly useful for identifying dynamic objects against a relatively static background.

In [22]:
def recursive_moving_average(video_path: str, alpha: float = 0.1, width: int = 500):
    """
    Applies Recursive Moving Average (RMA) for motion detection in a video.

    Parameters:
    - video_path: str -> Path to the video file.
    - alpha: float -> Smoothing factor that controls adaptation speed (default: 0.1).
    - width: int -> Width to resize frames for consistency (default: 500).
    """
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error opening video file")
        return

    # Read the first frame
    _, prev_frame = cap.read()
    prev_frame = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    prev_frame = imutils.resize(prev_frame, width=width)
    avg = np.float32(prev_frame)  # Initialize background model

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = imutils.resize(frame, width=width)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Update the background model
        avg = (1 - alpha) * avg + alpha * gray
        avg_display = cv2.convertScaleAbs(avg)

        # Compute absolute difference
        diff = cv2.absdiff(cv2.convertScaleAbs(gray), avg_display)

        # Apply thresholding
        _, thresh = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)

        # Display results
        cv2.imshow('Frame', gray)
        cv2.imshow('Static Background', avg_display)
        cv2.imshow('Difference', diff)
        cv2.imshow('Motion Mask', thresh)

        # Wait for user input to progress frame-by-frame
        key = cv2.waitKey(0) & 0xFF
        if key == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

In [41]:
recursive_moving_average("data/168811-839864556_tiny.mp4", alpha=0.2)

- Motion is clearly detected in the difference image and thresholded motion mask.
- Small values of $ \alpha $ result in a smoother average frame, leading to slower adaptation to changes. This helps preserve the background for longer, making it useful for reducing noise or detecting slow-moving objects.
- Larger values of $ \alpha $ adapt quickly to changes, making the algorithm more responsive to fast-moving objects. However, this can cause stationary objects that remain for a short duration to fade from the background.

## 2. Motion Detection Using the Richefeu Algorithm
The Richefeu algorithm is a motion detection approach based on the morphological evolution of pixels in a video sequence. It uses two adaptive smoothing filters:

- **Morphological Ouverture Oublieuse (MOO)**: This step updates the maximum pixel values adaptively to model dynamic background changes and reduce false positives.

- **Morphological Fermeture Oublieuse (MFO)**: This step updates the minimum pixel values to capture intensity variations and minimize the effects of local lighting changes.

The difference between these two models produces a **morphological gradient**, highlighting moving regions. A thresholding operation is then applied to extract significant motion areas.

In [39]:
def richefeu_motion_detection(video_path: str, alpha: float = 0.45, delta: int = 40, taux: int = 1, width: int = 600, min_contour_area: int = 500):
    """
    Applies Richefeu's motion detection algorithm to a video.

    Parameters:
    - video_path: str -> Path to the video file.
    - alpha: float -> Forgetting factor controlling adaptation speed (default: 0.45).
    - delta: int -> Threshold for motion detection (default: 40).
    - taux: int -> Number of frames skipped between analyses (default: 1).
    - width: int -> Width to resize frames for consistency (default: 600).
    - min_contour_area (int): Minimum contour area for motion detection.
    """
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error opening video file")
        return

    # Read the first frame
    ret, frame = cap.read()
    if not ret:
        print("Error reading the first frame")
        cap.release()
        return

    frame = imutils.resize(frame, width=width)
    Mt = mt = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32)

    while True:
        for _ in range(taux):
            ret, It = cap.read()
            if not ret:
                break

        if not ret:
            break

        It = cv2.cvtColor(It, cv2.COLOR_BGR2GRAY).astype(np.float32)
        It = imutils.resize(It, width=width)

        # Morphological Opening Forgetting (MOO) - Adaptive Dilation
        Mt = alpha * It + (1 - alpha) * cv2.max(It, Mt)

        # Morphological Closing Forgetting (MFO) - Adaptive Erosion
        mt = alpha * It + (1 - alpha) * cv2.min(It, mt)

        # Morphological Gradient
        Gamma_t = Mt - mt
        D_t = np.where(Gamma_t >= delta, 255, 0).astype(np.uint8)

        # Dilate to enhance detection
        D_t = cv2.dilate(D_t, None, iterations=1)

        # Convert to uint8 for visualization
        It_display = It.astype(np.uint8)
        Gamma_display = Gamma_t.astype(np.uint8)

        # Find contours
        cnts = cv2.findContours(D_t.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cnts = imutils.grab_contours(cnts)

        for c in cnts:
            if cv2.contourArea(c) < min_contour_area:
                continue
            # Compute the bounding box for the contour and draw it on the frame
            x, y, w, h = cv2.boundingRect(c)
            cv2.rectangle(It_display, (x, y), (x + w, y + h), (0, 255, 0), 2)

        # Display results
        cv2.imshow('Input Frame', It_display)
        cv2.imshow('Gradient (Gamma_t)', Gamma_display)
        cv2.imshow('Detected Motion (D_t)', D_t)

        key = cv2.waitKey(0) & 0xFF
        if key == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

In [ ]:
richefeu_motion_detection("data/168811-839864556_tiny.mp4", alpha=0.2, delta=30, min_contour_area=300)

- The algorithm effectively isolates moving objects while removing static noise.
- The contours of detected objects are well-defined, facilitating motion region extraction.
- A well-chosen threshold is crucial to balance between avoiding false positives (minor light changes) and false negatives (low-contrast objects).
- The parameter $\alpha $ controls background adaptation, similar to RMA. Smaller values preserve static backgrounds, while larger values respond faster to motion.
- The parameter $\tau $ controls the frame skipping rate, allowing motion analysis over larger temporal gaps. Higher values make the algorithm less sensitive to short-term motion but better suited for detecting long-term movements.